# Week 20, Day 5: Channels, efficient coding and the information bottleneck

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w020/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Information from correlation

For signal-to-noise ratios 0.25, 1, 4 and 16, simulate 200,000 samples of a unit-variance Gaussian signal plus noise of variance 1/SNR, estimate the correlation between signal and response, and convert it to information with −½ log₂(1 − ρ²). Store the results in `info_from_corr`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2000)
def entropy(p):
    p = np.asarray(p, float); p = p[p > 0]
    return -np.sum(p * np.log2(p))

snrs = np.array([0.25, 1, 4, 16])
info_from_corr = []
for snr in snrs:
    s = rng.normal(size=200000)
    r = s + rng.normal(scale=np.sqrt(1 / snr), size=200000)
    rho = ...
    info_from_corr.append(...)
info_from_corr = np.array(info_from_corr)
print("SNR:", snrs, " information from correlation:", info_from_corr.round(3), " formula:", (0.5 * np.log2(1 + snrs)).round(3))

In [ ]:
import numpy as np
def test_formula():
    "for each SNR the information from the sample correlation is within 0.03 bits of 0.5 log2(1 + SNR)"
    assert np.all(np.abs(info_from_corr - 0.5 * np.log2(1 + snrs)) < 0.03)

test_formula()
print("All checks passed")

### Exercise: Two neurons, one signal

Two neurons respond to a unit-variance signal, each with independent noise of variance 1/3. Estimate the information of one neuron (`i_single`) and of the pair, using their average as the combined response (`i_pair`), from correlations as above. Compute `redundancy` = 2 × i_single − i_pair.

In [ ]:
s = rng.normal(size=200000)
r1 = s + rng.normal(scale=np.sqrt(1 / 3), size=200000)
r2 = s + rng.normal(scale=np.sqrt(1 / 3), size=200000)
def info_corr(a, b): rho = np.corrcoef(a, b)[0, 1]; return -0.5 * np.log2(1 - rho ** 2)
i_single = ...
i_pair = ...
redundancy = 2 * i_single - i_pair
print(f"one neuron {i_single:.3f} bits, both {i_pair:.3f} bits, redundancy {redundancy:.3f} bits")

In [ ]:
import numpy as np
def test_single():
    "each neuron alone carries about 1 bit (within 0.03)"
    assert abs(i_single - 1) < 0.03

def test_pair():
    "together they carry about 1.40 bits, so redundancy is about 0.6 bits"
    assert abs(i_pair - 0.5 * np.log2(7)) < 0.03 and abs(redundancy - (2 - 0.5 * np.log2(7))) < 0.05

test_single()
test_pair()
print("All checks passed")

### Exercise: The best two-word summary

X takes 8 equally likely values; Y is binary with p(Y = 1 | x) given by `p_y1`. Try every split of the 8 values into two non-empty groups (T = 0 or 1), compute I(T; Y) for each, and keep the split with the largest. Store `best_part` (an array of 0s and 1s), `ity_best`, `ixt_best` = I(X; T) and `ixy` = I(X; Y).

In [ ]:
from itertools import product
p_x = np.full(8, 1 / 8)
p_y1 = np.array([0.05, 0.1, 0.12, 0.2, 0.75, 0.8, 0.9, 0.95])
joint_xy = np.column_stack([p_x * (1 - p_y1), p_x * p_y1])      # 8 x 2

def mi(J):
    J = np.asarray(J, float)
    return entropy(J.sum(1)) + entropy(J.sum(0)) - entropy(J.ravel())

ixy = mi(joint_xy)
best = (-1, None)
for bits in product([0, 1], repeat=8):
    part = np.array(bits)
    if part.sum() in (0, 8):
        continue
    joint_ty = ...                                  # 2 x 2: rows T, columns Y
    ity = mi(joint_ty)
    if ity > best[0]:
        best = (ity, part)
ity_best, best_part = best
ixt_best = entropy([p_x[best_part == 0].sum(), p_x[best_part == 1].sum()])   # T is a deterministic function of X
print(f"I(X; Y) = {ixy:.3f} bits;  best two-cluster summary keeps {ity_best:.3f} bits using I(X; T) = {ixt_best:.3f} bits;  clusters {best_part}")

In [ ]:
import numpy as np
def test_ixt():
    "a deterministic two-cluster summary has I(X; T) at most 1 bit"
    assert ixt_best <= 1 + 1e-9

def test_keeps():
    "the best partition keeps at least 85% of I(X; Y)"
    assert ity_best >= 0.85 * ixy

def test_groups():
    "it groups the x values by their probability of Y = 1: every x in the higher group has p(y = 1 | x) above every x in the lower group"
    hi, lo = p_y1[best_part == 1], p_y1[best_part == 0]
    if hi.size and lo.size: assert min(hi.min(), lo.min()) == min(hi.min(), lo.min()) and (hi.min() > lo.max() or lo.min() > hi.max())

test_ixt()
test_keeps()
test_groups()
print("All checks passed")